In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import NearestNeighbors
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
data_path = 'data/light_spotify_dataset.csv'
df = pd.read_csv(data_path)

In [3]:
# Create a copy for preprocessing
df_processed = df.copy()

# Drop rows with missing critical values
df_processed = df_processed.dropna(subset=['artist', 'song'])

# Fill missing numerical values with median
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    df_processed[col] = df_processed[col].fillna(df_processed[col].median())

# Fill missing categorical values with 'unknown'
categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_processed[col] = df_processed[col].fillna('unknown')

In [4]:
columns=df_processed.columns.tolist()   
print(columns)

categorical_cols = df_processed.select_dtypes(include=['object']).columns
print(categorical_cols)

['artist', 'song', 'emotion', 'variance', 'Genre', 'Release Date', 'Key', 'Tempo', 'Loudness', 'Explicit', 'Popularity', 'Energy', 'Danceability', 'Positiveness', 'Speechiness', 'Liveness', 'Acousticness', 'Instrumentalness']
Index(['artist', 'song', 'emotion', 'Genre', 'Key', 'Explicit'], dtype='str')


In [5]:
# Select audio features for recommendation
audio_features = ['Energy', 'Danceability', 'Positiveness', 'Speechiness', 
                'Liveness', 'Acousticness', 'Instrumentalness', 'Tempo', 
                'Loudness', 'Popularity']

# Create feature matrix with audio features
X = df_processed[audio_features].copy()

# Add encoded categorical features
genre_encoded = pd.get_dummies(df_processed['Genre'], prefix='genre', drop_first=True)
emotion_encoded = pd.get_dummies(df_processed['emotion'], prefix='emotion', drop_first=True)
explicit_encoded = (df_processed['Explicit'] == 'Yes').astype(int).values.reshape(-1, 1)

# Combine all features
X = pd.concat([X, genre_encoded, emotion_encoded], axis=1)
X = np.column_stack([X, explicit_encoded])


In [6]:

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled data shape: {X_scaled.shape}")

Scaled data shape: (236980, 2581)


In [7]:
n_neighbors = 10
knn_model = NearestNeighbors(n_neighbors=n_neighbors, algorithm='auto', metric='cosine')
knn_model.fit(X_scaled)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [8]:
def get_recommendations(song_name=None, song_index=None, num_recommendations=5):
    
    # Find song index if name is provided
    if song_name is not None:
        matching_songs = df_processed[df_processed['song'].str.lower() == song_name.lower()]
        if len(matching_songs) == 0:
            print(f"Song '{song_name}' not found in dataset.")
            return None
        song_index = matching_songs.index[0]
    
    if song_index is None:
        print("Please provide either song_name or song_index")
        return None
    
    # Check if index is valid
    if song_index >= len(X_scaled):
        print(f"Invalid song index: {song_index}")
        return None
    
    # Get distances and indices of nearest neighbors
    distances, indices = knn_model.kneighbors([X_scaled[song_index]], n_neighbors=num_recommendations+1)
    
    # Remove the first entry (the song itself) and get recommendations
    recommendation_indices = indices[0][1:num_recommendations+1]
    recommendation_distances = distances[0][1:num_recommendations+1]
    
    # Get song details
    recommended_songs = df_processed.iloc[recommendation_indices][['artist', 'song', 'Genre', 'emotion', 'Popularity']].copy()
    recommended_songs['similarity_score'] = (1 - recommendation_distances) * 100  # Convert to similarity percentage
    recommended_songs = recommended_songs.reset_index(drop=True)
    
    return recommended_songs

In [13]:

song_index = 50
song_name = df_processed.iloc[song_index]['song']
artist_name = df_processed.iloc[song_index]['artist']
print(f"Input Song: '{song_name}' by {artist_name}\n")

recommendations = get_recommendations(song_index=song_index, num_recommendations=10)
print("Top 10 Similar Songs:")
print(recommendations.to_string(index=True))

Input Song: 'Lay All Your Love On Me' by ABBA

Top 10 Similar Songs:
          artist                     song Genre emotion  Popularity  similarity_score
0           ABBA  Lay All Your Love on Me   pop   anger          78         99.993992
1           MIKA              Grace Kelly   pop   anger          73         97.478390
2   Cyndi Lauper       The More I See You   pop   anger          84         97.279800
3       Haddaway            What Is Love?   pop   anger          80         96.991731
4  Justin Bieber              Its Working   pop   anger          73         96.938386
5     Eurythmics           Damien Save Me   pop   anger          68         96.276566
6     Eurythmics         Crown Of Madness   pop   anger          68         96.276566
7    Lene Marlin        Sitting Down Here   pop   anger          62         96.191482
8     Bruno Mars      Dance in the Mirror   pop   anger          90         96.115012
9         MAGIC!                     Rude   pop   anger          84    

In [16]:

# Find a popular song
popular_songs = df_processed.nlargest(5, 'Popularity')
test_song_name = popular_songs.iloc[0]['song']
print(f"Searching for: '{test_song_name}'\n")

recommendations2 = get_recommendations(song_name=test_song_name, num_recommendations=8)
if recommendations2 is not None:
    print("Top 8 Similar Songs:")
    print(recommendations2.to_string(index=True))


Searching for: 'Maybe The Night'

Top 8 Similar Songs:
            artist                  song                    Genre emotion  Popularity  similarity_score
0          Ben&Ben              Lifetime  Unknown,Unknown,Unknown     joy         100        100.000000
1          Ben&Ben      Make It With You  Unknown,Unknown,Unknown     joy         100        100.000000
2          Ben&Ben                  Fall  Unknown,Unknown,Unknown     joy         100        100.000000
3          Ben&Ben                 Doors  Unknown,Unknown,Unknown     joy         100        100.000000
4          Ben&Ben                   War  Unknown,Unknown,Unknown     joy         100        100.000000
5          Ben&Ben       Maybe The Night  Unknown,Unknown,Unknown     joy         100        100.000000
6  BEAUZ & SIIGHTS  Never Gonna Regret U  Unknown,Unknown,Unknown     joy          99         99.996474
7      Chas & Dave                Rabbit  Unknown,Unknown,Unknown     joy          83         96.034835


In [17]:
import random
random_index = random.randint(0, len(df_processed) - 1)
random_song = df_processed.iloc[random_index]['song']
random_artist = df_processed.iloc[random_index]['artist']

recommendations3 = get_recommendations(song_index=random_index, num_recommendations=5)
print("Top 5 Similar Songs:")
print(recommendations3.to_string(index=True))

Top 5 Similar Songs:
              artist                           song Genre emotion  Popularity  similarity_score
0         Eartheater                      High Tide  folk     joy          46         97.827300
1  Little Green Cars                 The John Wayne  folk     joy          38         97.784072
2  Bruce Springsteen            If I Was the Priest  folk     joy          39         97.729541
3        Bright Eyes                 Sunrise Sunset  folk     joy          32         97.641157
4   The Decemberists  Rusalka Rusalka / Wild Rushes  folk     joy          30         97.533159


In [9]:
import os
import json

# Create model artifacts directory
model_dir = './model_artifacts'
os.makedirs(model_dir, exist_ok=True)

# Save the KNN model
model_path = os.path.join(model_dir, 'music_recommendation_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(knn_model, f)

# Save the scaler
scaler_path = os.path.join(model_dir, 'feature_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

# Save the processed dataset (for reference and mapping indices to song info)
dataset_path = os.path.join(model_dir, 'music_dataset.csv')
df_processed.to_csv(dataset_path, index=False)

# Save feature names for reconstruction
feature_names = (audio_features + list(genre_encoded.columns) + 
                list(emotion_encoded.columns) + 
                ['explicit_flag'])

features_metadata = {
    'audio_features': audio_features,
    'num_genre_features': genre_encoded.shape[1],
    'num_emotion_features': emotion_encoded.shape[1],
    'total_features': len(feature_names),
    'n_neighbors': n_neighbors,
    'metric': 'cosine',
    'algorithm': 'auto'
}

metadata_path = os.path.join(model_dir, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(features_metadata, f, indent=4)